In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 97
==================================================
Week: 14 of 24
Day: 97 of 168
Date: May 12, 2026 
Topic: Complete Test Suite & Performance Testing

Week 14 Progress:
✅ Day 92: Docker Fundamentals
✅ Day 93: Dockerize Security System
✅ Day 94: RESTful API Design & FastAPI Deep Dive
✅ Day 95: API Authentication & Security
✅ Day 96: Building a CI/CD Pipeline
🔄 Day 97: Complete Test Suite & Performance Testing (TODAY!)
⬜ Day 98: Model Monitoring & Logging

Progress: 71.4% (5/7 days)

==================================================
🎯 Week 14 Project: MLOps & Production Engineering
- Containerize ML projects with Docker
- Build production-grade APIs with FastAPI
- Automate deployments with GitHub Actions CI/CD
- Add monitoring, logging, and error tracking

🎯 Today's Learning Objectives:
1. Master advanced pytest patterns (fixtures, parametrize, markers)
2. Implement test coverage reporting with pytest-cov
3. Write integration tests with real Redis
4. Performance test the API with Locust
5. Finalize the complete test suite for the Security System

📚 Today's Structure:
   Part 1 (1.5h): Advanced Pytest Patterns
   Part 2 (1.5h): Test Coverage Reporting
   Part 3 (1.5h): Integration Testing with Redis
   Part 4 (1.5h): Performance Testing with Locust

🎯 SUCCESS CRITERIA:
   ✅ Write parametrized and marked tests for the Security System
   ✅ Achieve 80%+ test coverage and generate HTML coverage report
   ✅ Write integration tests that spin up real Redis via Docker
   ✅ Run Locust load test and analyze API performance under load
   ✅ Complete test suite ready for CI pipeline

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
!{sys.executable} -m pip install pytest pytest-asyncio pytest-cov pytest-mock httpx locust fakeredis -q

print("✅ Libraries installed!")

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

import os
import time
import json
import asyncio
from unittest.mock import MagicMock, patch, AsyncMock
from datetime import datetime

print("\n✅ All libraries imported successfully!")
print("=" * 80)

✅ Libraries installed!

📚 IMPORTING LIBRARIES

✅ All libraries imported successfully!


'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
print("\n" + "=" * 80)
print("🧪 PART 1: ADVANCED PYTEST PATTERNS")
print("=" * 80)


🧪 PART 1: ADVANCED PYTEST PATTERNS


In [3]:
# ==================================================
# EXERCISE 1.1: PYTEST FIXTURES DEEP DIVE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Pytest Fixtures — Scopes and Teardown")
print("=" * 80)

"""
📖 THEORY: Fixture Scopes

Fixtures have 4 scopes that control how often they're created:
- function  → created fresh for EACH test (default)
- class     → shared within a test class
- module    → shared within a test file
- session   → shared across the ENTIRE test run

Rule: use the widest scope that's safe.
      Expensive setup (DB connection, model load) → session
      State that changes between tests → function
"""

advanced_fixtures_py = '''
# ============================================================
# tests/conftest.py — Advanced Fixtures
# ============================================================

import pytest
import fakeredis
import asyncio
from fastapi.testclient import TestClient
from unittest.mock import MagicMock, AsyncMock
from api.main import app
from api.auth.jwt import create_access_token


# ── Scope: session — expensive setup, created ONCE ───────────
@pytest.fixture(scope="session")
def mock_model():
    """
    Mock YOLO model — session scope means it\'s created once
    and reused across ALL tests. Avoids repeated mock setup overhead.
    """
    model = MagicMock()

    # Simulate a real YOLO result structure
    mock_box = MagicMock()
    mock_box.xyxy = [[10.0, 20.0, 100.0, 200.0]]
    mock_box.conf = [0.92]
    mock_box.cls = [0]

    mock_result = MagicMock()
    mock_result.boxes = [mock_box]
    mock_result.names = {0: "person"}
    model.return_value = [mock_result]

    return model


@pytest.fixture(scope="session")
def client(mock_model):
    """Test client — session scope, shared across all tests."""
    app.state.model = mock_model
    with TestClient(app, raise_server_exceptions=False) as c:
        yield c


# ── Scope: function — fresh state per test ────────────────────
@pytest.fixture
def fake_redis():
    """
    In-memory Redis using fakeredis — no real Redis needed.
    function scope: each test gets a CLEAN Redis instance.
    """
    server = fakeredis.FakeServer()
    redis = fakeredis.FakeRedis(server=server, decode_responses=True)
    yield redis
    redis.flushall()   # Teardown: clean up after test


# ── Auth token fixtures ───────────────────────────────────────
@pytest.fixture
def viewer_token():
    return create_access_token(user_id="user-viewer-001", role="viewer")

@pytest.fixture
def operator_token():
    return create_access_token(user_id="user-operator-001", role="operator")

@pytest.fixture
def admin_token():
    return create_access_token(user_id="user-admin-001", role="admin")

@pytest.fixture
def viewer_headers(viewer_token):
    return {"Authorization": f"Bearer {viewer_token}"}

@pytest.fixture
def operator_headers(operator_token):
    return {"Authorization": f"Bearer {operator_token}"}

@pytest.fixture
def admin_headers(admin_token):
    return {"Authorization": f"Bearer {admin_token}"}


# ── Sample data fixtures ──────────────────────────────────────
@pytest.fixture
def sample_detection_response():
    """Pre-built detection response for tests that need one."""
    return {
        "id": "test-detection-001",
        "status": "complete",
        "filename": "test.jpg",
        "frame_count": 1,
        "detections": [
            {
                "track_id": None,
                "class_name": "person",
                "confidence": 0.92,
                "bounding_box": {"x1": 10.0, "y1": 20.0, "x2": 100.0, "y2": 200.0}
            }
        ],
        "processing_time_ms": 45.2,
        "model_version": "yolov8n",
    }
'''

print("📄 tests/conftest.py (advanced):")
print(advanced_fixtures_py)

print("""
💡 FIXTURE TEARDOWN PATTERNS:

  # Yield-based (recommended — runs cleanup after test)
  @pytest.fixture
  def fake_redis():
      redis = fakeredis.FakeRedis()
      yield redis          ← test runs here
      redis.flushall()     ← teardown runs after test

  # Finalize-based (alternative)
  @pytest.fixture
  def temp_file(tmp_path):
      f = tmp_path / "test.json"
      f.write_text("{}")
      yield f
      f.unlink()           ← cleanup

  # tmp_path is a built-in pytest fixture — gives you a
  # temporary directory unique to each test, auto-cleaned up
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Pytest Fixtures — Scopes and Teardown
📄 tests/conftest.py (advanced):

# ============================================================
# tests/conftest.py — Advanced Fixtures
# ============================================================

import pytest
import fakeredis
import asyncio
from fastapi.testclient import TestClient
from unittest.mock import MagicMock, AsyncMock
from api.main import app
from api.auth.jwt import create_access_token


# ── Scope: session — expensive setup, created ONCE ───────────
@pytest.fixture(scope="session")
def mock_model():
    """
    Mock YOLO model — session scope means it's created once
    and reused across ALL tests. Avoids repeated mock setup overhead.
    """
    model = MagicMock()

    # Simulate a real YOLO result structure
    mock_box = MagicMock()
    mock_box.xyxy = [[10.0, 20.0, 100.0, 200.0]]
    mock_box.conf = [0.92]
    mock_box.cls = [0]

    mock_result = MagicMock()
    mock_result.boxes = [mock_box]
    mock_result.na

In [4]:
# ==================================================
# EXERCISE 1.2: PARAMETRIZE AND MARKERS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Parametrize and Custom Markers")
print("=" * 80)

"""
📖 THEORY: Parametrize

@pytest.mark.parametrize lets you run the SAME test with
multiple inputs — avoids copy-pasting test functions.

Custom markers let you group tests and run subsets:
  pytest tests/ -m "unit"          → only unit tests
  pytest tests/ -m "not slow"      → skip slow tests
  pytest tests/ -m "auth"          → only auth tests
"""

parametrize_tests_py = '''
# ============================================================
# tests/test_detections_parametrized.py
# ============================================================

import pytest
import io
from PIL import Image


def make_image(format="JPEG") -> bytes:
    img = Image.new("RGB", (640, 480), color=(128, 128, 128))
    buf = io.BytesIO()
    img.save(buf, format=format)
    return buf.getvalue()


# ── Parametrize: test multiple confidence values ──────────────
@pytest.mark.parametrize("confidence,expected_status", [
    (0.1,  201),   # valid — minimum
    (0.5,  201),   # valid — default
    (0.99, 201),   # valid — near maximum
    (0.0,  422),   # invalid — below minimum (ge=0.1)
    (1.5,  422),   # invalid — above maximum (le=1.0)
    (-0.1, 422),   # invalid — negative
])
def test_detection_confidence_validation(
    client, operator_headers, confidence, expected_status
):
    """Test confidence parameter validation across boundary values."""
    image_bytes = make_image()
    response = client.post(
        f"/api/v1/detections?confidence={confidence}",
        files={"file": ("test.jpg", image_bytes, "image/jpeg")},
        headers=operator_headers,
    )
    assert response.status_code == expected_status, (
        f"confidence={confidence} expected {expected_status}, "
        f"got {response.status_code}"
    )


# ── Parametrize: test file type validation ────────────────────
@pytest.mark.parametrize("filename,content_type,expected_status", [
    ("test.jpg",  "image/jpeg",       201),   # valid
    ("test.png",  "image/png",        201),   # valid
    ("test.txt",  "text/plain",       400),   # invalid
    ("test.pdf",  "application/pdf",  400),   # invalid
    ("test.exe",  "application/octet-stream", 400),  # invalid
])
def test_detection_file_type_validation(
    client, operator_headers, filename, content_type, expected_status
):
    """Test file type validation for detection uploads."""
    image_bytes = make_image() if "image" in content_type else b"fake content"
    response = client.post(
        "/api/v1/detections",
        files={"file": (filename, image_bytes, content_type)},
        headers=operator_headers,
    )
    assert response.status_code == expected_status


# ── Markers: group tests by category ─────────────────────────
@pytest.mark.unit
def test_bounding_box_area_calculation():
    """Unit test — no HTTP, no fixtures needed."""
    from api.schemas.detection import BoundingBox
    box = BoundingBox(x1=0, y1=0, x2=100, y2=50)
    assert box.area == 5000.0


@pytest.mark.unit
def test_bounding_box_x2_must_exceed_x1():
    """Pydantic validator should reject x2 <= x1."""
    from api.schemas.detection import BoundingBox
    from pydantic import ValidationError
    with pytest.raises(ValidationError):
        BoundingBox(x1=100, y1=0, x2=50, y2=100)   # x2 < x1


@pytest.mark.auth
@pytest.mark.parametrize("role,endpoint,expected", [
    ("viewer",   "GET /api/v1/detections",         200),
    ("viewer",   "POST /api/v1/detections",         403),  # operator+ only
    ("operator", "POST /api/v1/detections",         201),
    ("operator", "DELETE /api/v1/detections/123",   403),  # admin only
    ("admin",    "DELETE /api/v1/detections/123",   404),  # admin, but 404 (not found)
])
def test_role_based_access(client, viewer_headers, operator_headers,
                            admin_headers, role, endpoint, expected):
    """Test RBAC — each role can only access what they\'re allowed to."""
    headers_map = {
        "viewer": viewer_headers,
        "operator": operator_headers,
        "admin": admin_headers,
    }
    method, path = endpoint.split(" ", 1)
    headers = headers_map[role]

    if method == "GET":
        response = client.get(path, headers=headers)
    elif method == "POST":
        from tests.conftest import make_image
        response = client.post(
            path,
            files={"file": ("t.jpg", make_image(), "image/jpeg")},
            headers=headers
        )
    elif method == "DELETE":
        response = client.delete(path, headers=headers)

    assert response.status_code == expected
'''

pytest_ini = '''
# ============================================================
# pytest.ini — Pytest configuration
# ============================================================

[pytest]
asyncio_mode = auto
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*

# Custom markers (avoids PytestUnknownMarkWarning)
markers =
    unit: pure unit tests with no external dependencies
    integration: tests requiring external services (Redis, DB)
    auth: authentication and authorization tests
    slow: tests that take more than 5 seconds
    performance: load and performance tests (run separately)

# Default options
addopts =
    -v
    --tb=short
    --strict-markers
'''

print("📄 tests/test_detections_parametrized.py:")
print(parametrize_tests_py)
print("\n📄 pytest.ini:")
print(pytest_ini)

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Parametrize and Custom Markers
📄 tests/test_detections_parametrized.py:

# ============================================================
# tests/test_detections_parametrized.py
# ============================================================

import pytest
import io
from PIL import Image


def make_image(format="JPEG") -> bytes:
    img = Image.new("RGB", (640, 480), color=(128, 128, 128))
    buf = io.BytesIO()
    img.save(buf, format=format)
    return buf.getvalue()


# ── Parametrize: test multiple confidence values ──────────────
@pytest.mark.parametrize("confidence,expected_status", [
    (0.1,  201),   # valid — minimum
    (0.5,  201),   # valid — default
    (0.99, 201),   # valid — near maximum
    (0.0,  422),   # invalid — below minimum (ge=0.1)
    (1.5,  422),   # invalid — above maximum (le=1.0)
    (-0.1, 422),   # invalid — negative
])
def test_detection_confidence_validation(
    client, operator_headers, confidence, expected_status
):
    """Test confide

In [5]:
print("\n" + "=" * 80)
print("📊 PART 2: TEST COVERAGE REPORTING")
print("=" * 80)


📊 PART 2: TEST COVERAGE REPORTING


In [6]:
# ==================================================
# EXERCISE 2.1: PYTEST-COV SETUP
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Test Coverage with pytest-cov")
print("=" * 80)

"""
📖 THEORY: Test Coverage

Coverage measures what % of your source code is executed
during the test run. High coverage = more confidence in tests.

Coverage types:
- Line coverage    → was this line executed?
- Branch coverage  → were both True/False branches taken?
- Function coverage → was this function called?

Target for ML APIs: 80%+ line coverage, 70%+ branch coverage.
100% is rarely worth chasing — focus on critical paths.
"""

print("""
📋 RUNNING COVERAGE:

  # Basic coverage report (terminal)
  pytest tests/ --cov=api --cov-report=term-missing

  # HTML report (open in browser)
  pytest tests/ --cov=api --cov-report=html

  # Both terminal + HTML
  pytest tests/ --cov=api --cov-report=term-missing --cov-report=html

  # Fail if coverage drops below 80%
  pytest tests/ --cov=api --cov-fail-under=80

  # Exclude files from coverage
  pytest tests/ --cov=api --cov-omit="api/core/logger.py,api/migrations/*"

  HTML report generated at: htmlcov/index.html
  → Shows each file with green (covered) and red (uncovered) lines
""")

coveragerc = '''
# ============================================================
# .coveragerc — Coverage configuration
# ============================================================

[run]
source = api
omit =
    api/__init__.py
    api/migrations/*
    api/core/logger.py        # Logging setup — not critical to test
    tests/*

[report]
exclude_lines =
    pragma: no cover          # Skip lines marked with this comment
    def __repr__
    raise NotImplementedError
    if TYPE_CHECKING:
    pass

[html]
directory = htmlcov
title = AI Security System — Coverage Report

[xml]
output = coverage.xml         # For CI/CD tools (Codecov, SonarQube)
'''

print("📄 .coveragerc:")
print(coveragerc)

print("""
📋 COVERAGE IN CI PIPELINE (update ci.yml):

  - name: Run tests with coverage
    env:
      REDIS_URL: redis://localhost:6379/0
      SECRET_KEY: test-secret-key
      API_KEY: test-api-key
    run: |
      pytest tests/ -v --tb=short \\
        --cov=api \\
        --cov-report=term-missing \\
        --cov-report=xml \\
        --cov-fail-under=80

  - name: Upload coverage to Codecov
    uses: codecov/codecov-action@v4
    with:
      file: coverage.xml
      fail_ci_if_error: false

  - name: Upload HTML coverage report
    uses: actions/upload-artifact@v4
    with:
      name: coverage-report
      path: htmlcov/
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Test Coverage with pytest-cov

📋 RUNNING COVERAGE:

  # Basic coverage report (terminal)
  pytest tests/ --cov=api --cov-report=term-missing

  # HTML report (open in browser)
  pytest tests/ --cov=api --cov-report=html

  # Both terminal + HTML
  pytest tests/ --cov=api --cov-report=term-missing --cov-report=html

  # Fail if coverage drops below 80%
  pytest tests/ --cov=api --cov-fail-under=80

  # Exclude files from coverage
  pytest tests/ --cov=api --cov-omit="api/core/logger.py,api/migrations/*"

  HTML report generated at: htmlcov/index.html
  → Shows each file with green (covered) and red (uncovered) lines

📄 .coveragerc:

# ============================================================
# .coveragerc — Coverage configuration
# ============================================================

[run]
source = api
omit =
    api/__init__.py
    api/migrations/*
    api/core/logger.py        # Logging setup — not critical to test
    tests/*

[report]
exclude_lines =
    p

In [7]:
# ==================================================
# EXERCISE 2.2: WHAT TO TEST FOR 80% COVERAGE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Coverage Strategy for the Security System")
print("=" * 80)

print("""
📊 COVERAGE TARGETS BY MODULE:

  Module                        Target    Priority
  ──────────────────────────────────────────────────────────
  api/routers/detections.py     90%+      🔴 Critical — core feature
  api/routers/auth.py           90%+      🔴 Critical — security
  api/routers/cameras.py        80%+      🟡 Important
  api/schemas/detection.py      95%+      🔴 Critical — validation logic
  api/auth/jwt.py               90%+      🔴 Critical — security
  api/auth/api_key.py           90%+      🔴 Critical — security
  api/core/dependencies.py      80%+      🟡 Important
  api/core/middleware.py        70%+      🟢 Nice to have
  api/core/exceptions.py        80%+      🟡 Important
  api/services/detection.py     85%+      🔴 Critical — ML inference logic
  api/core/config.py            60%+      🟢 Simple env var loading

📋 WHAT NOT TO TEST:

  ❌ Third-party library internals (FastAPI, Pydantic, Redis)
  ❌ Pure configuration (settings values)
  ❌ Abstract base classes with no logic
  ❌ __repr__ and __str__ methods
  ❌ Code paths that only run in Python itself (TYPE_CHECKING blocks)

📋 WHAT MUST BE TESTED:

  ✅ Every API endpoint (happy path + error cases)
  ✅ All Pydantic validators (valid input + boundary values)
  ✅ Auth flows (valid token, expired token, wrong role)
  ✅ Rate limiting (429 after limit exceeded)
  ✅ Exception handlers (ensure clean error responses)
  ✅ Service layer business logic (detection, caching)
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Coverage Strategy for the Security System

📊 COVERAGE TARGETS BY MODULE:

  Module                        Target    Priority
  ──────────────────────────────────────────────────────────
  api/routers/detections.py     90%+      🔴 Critical — core feature
  api/routers/auth.py           90%+      🔴 Critical — security
  api/routers/cameras.py        80%+      🟡 Important
  api/schemas/detection.py      95%+      🔴 Critical — validation logic
  api/auth/jwt.py               90%+      🔴 Critical — security
  api/auth/api_key.py           90%+      🔴 Critical — security
  api/core/dependencies.py      80%+      🟡 Important
  api/core/middleware.py        70%+      🟢 Nice to have
  api/core/exceptions.py        80%+      🟡 Important
  api/services/detection.py     85%+      🔴 Critical — ML inference logic
  api/core/config.py            60%+      🟢 Simple env var loading

📋 WHAT NOT TO TEST:

  ❌ Third-party library internals (FastAPI, Pydantic, Redis)
  ❌ Pure configuration (

In [8]:
print("\n" + "=" * 80)
print("🔗 PART 3: INTEGRATION TESTING WITH REDIS")
print("=" * 80)


🔗 PART 3: INTEGRATION TESTING WITH REDIS


In [9]:
# ==================================================
# EXERCISE 3.1: INTEGRATION TESTS WITH FAKEREDIS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Integration Tests with FakeRedis")
print("=" * 80)

"""
📖 THEORY: Integration vs Unit Tests

Unit test:        test one function in isolation — mock everything else
Integration test: test multiple components working TOGETHER

For Redis integration tests we have two options:
1. fakeredis  → in-memory Redis emulator (fast, no Docker needed)
2. Real Redis → spin up via Docker in CI (slower, more realistic)

Use fakeredis for most tests, real Redis for final integration smoke test.
"""

integration_tests_py = '''
# ============================================================
# tests/test_integration_redis.py — Redis Integration Tests
# ============================================================

import pytest
import json
import fakeredis
from api.services.detection_service import DetectionService


@pytest.fixture
def redis_client():
    """Fresh fakeredis instance for each test."""
    client = fakeredis.FakeRedis(decode_responses=True)
    yield client
    client.flushall()


@pytest.fixture
def detection_service(mock_model, redis_client):
    return DetectionService(model=mock_model, redis=redis_client)


class TestDetectionCaching:

    @pytest.mark.integration
    async def test_cache_and_retrieve_detection(
        self, detection_service, redis_client, sample_detection_response
    ):
        """Detection result should be cached and retrievable."""
        detection_id = "test-123"

        # Cache the result
        await detection_service.cache_result(
            detection_id, sample_detection_response
        )

        # Retrieve from cache
        cached = await detection_service.get_cached_result(detection_id)

        assert cached is not None
        assert cached["id"] == sample_detection_response["id"]
        assert cached["filename"] == sample_detection_response["filename"]

    @pytest.mark.integration
    async def test_cache_expires_after_ttl(
        self, detection_service, redis_client, sample_detection_response
    ):
        """Cached result should expire after TTL."""
        detection_id = "test-expire-123"

        # Cache with very short TTL for testing
        await redis_client.setex(
            f"detection:{detection_id}",
            1,                              # 1 second TTL
            json.dumps(sample_detection_response)
        )

        # Immediately available
        cached = await detection_service.get_cached_result(detection_id)
        assert cached is not None

        # After expiry — gone
        import asyncio
        await asyncio.sleep(1.1)
        expired = await detection_service.get_cached_result(detection_id)
        assert expired is None

    @pytest.mark.integration
    async def test_cache_miss_returns_none(self, detection_service):
        """Cache miss should return None, not raise an exception."""
        result = await detection_service.get_cached_result("nonexistent-id")
        assert result is None

    @pytest.mark.integration
    async def test_list_detections_from_cache(
        self, detection_service, redis_client, sample_detection_response
    ):
        """List endpoint should return all cached detections."""
        # Cache 3 detections
        for i in range(3):
            det = {**sample_detection_response, "id": f"det-{i}"}
            await redis_client.setex(
                f"detection:det-{i}", 3600, json.dumps(det)
            )

        results = await detection_service.list_detections(limit=10, offset=0)

        assert results["total"] >= 3
        assert len(results["items"]) >= 3

    @pytest.mark.integration
    async def test_delete_detection_removes_from_cache(
        self, detection_service, redis_client, sample_detection_response
    ):
        """Deleting a detection should remove it from cache."""
        detection_id = "delete-test-123"
        await redis_client.setex(
            f"detection:{detection_id}",
            3600,
            json.dumps(sample_detection_response)
        )

        # Verify it exists
        assert await detection_service.get_cached_result(detection_id) is not None

        # Delete it
        deleted = await detection_service.delete_detection(detection_id)
        assert deleted is True

        # Verify it\'s gone
        assert await detection_service.get_cached_result(detection_id) is None
'''

print("📄 tests/test_integration_redis.py:")
print(integration_tests_py)

print("""
💡 FAKEREDIS vs REAL REDIS IN CI:

  fakeredis:
  ✅ No Docker required — works in any environment
  ✅ Very fast (in-memory, no network)
  ✅ Perfect for unit/integration tests of service logic
  ❌ May miss edge cases in real Redis behavior

  Real Redis (via CI services:):
  ✅ Tests actual Redis behavior (TTL, persistence, clustering)
  ✅ Catches connection/config issues early
  ❌ Requires Docker on the runner
  ❌ Slightly slower

  Strategy:
  → Use fakeredis for ALL service-layer integration tests
  → Use real Redis (ci.yml services:) for API endpoint tests
    that hit Redis through the full HTTP stack
""")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Integration Tests with FakeRedis
📄 tests/test_integration_redis.py:

# ============================================================
# tests/test_integration_redis.py — Redis Integration Tests
# ============================================================

import pytest
import json
import fakeredis
from api.services.detection_service import DetectionService


@pytest.fixture
def redis_client():
    """Fresh fakeredis instance for each test."""
    client = fakeredis.FakeRedis(decode_responses=True)
    yield client
    client.flushall()


@pytest.fixture
def detection_service(mock_model, redis_client):
    return DetectionService(model=mock_model, redis=redis_client)


class TestDetectionCaching:

    @pytest.mark.integration
    async def test_cache_and_retrieve_detection(
        self, detection_service, redis_client, sample_detection_response
    ):
        """Detection result should be cached and retrievable."""
        detection_id = "test-123"

        # Cache the r

In [10]:
print("\n" + "=" * 80)
print("⚡ PART 4: PERFORMANCE TESTING WITH LOCUST")
print("=" * 80)


⚡ PART 4: PERFORMANCE TESTING WITH LOCUST


In [11]:
# ==================================================
# EXERCISE 4.1: LOCUST LOAD TESTING
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.1: Load Testing with Locust")
print("=" * 80)

"""
📖 THEORY: Load Testing

Load testing simulates many concurrent users hitting your API.
It tells you:
- How many requests/second can your API handle?
- What's the average/p95/p99 response time under load?
- At what point does the API start failing or slowing down?
- Does the ML model become a bottleneck at scale?

Locust is Python-based load testing tool — tests written in Python,
not YAML or XML like JMeter.
"""

locustfile_py = '''
# ============================================================
# tests/performance/locustfile.py — Load Test Scenarios
# ============================================================

from locust import HttpUser, task, between, events
import io
import json
import random
from PIL import Image


def make_test_image(width=640, height=480) -> bytes:
    """Generate a test image for upload tasks."""
    img = Image.new("RGB", (width, height), color=(
        random.randint(0, 255),
        random.randint(0, 255),
        random.randint(0, 255)
    ))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


class SecuritySystemUser(HttpUser):
    """
    Simulates a typical Security System API user.
    wait_time: each user waits 1-3 seconds between requests
    (simulates realistic user behavior, not a DoS attack)
    """
    wait_time = between(1, 3)

    def on_start(self):
        """Called when a simulated user starts — authenticate first."""
        response = self.client.post("/auth/login", data={
            "username": "loadtest_user",
            "password": "loadtest_password"
        })
        if response.status_code == 200:
            token = response.json()["access_token"]
            self.headers = {"Authorization": f"Bearer {token}"}
        else:
            self.headers = {}

    # ── Read-heavy tasks (weight=3 = 3x more likely than weight=1) ──
    @task(3)
    def list_detections(self):
        """Most common operation — list recent detections."""
        self.client.get(
            "/api/v1/detections?limit=20&offset=0",
            headers=self.headers,
            name="/api/v1/detections [GET]"
        )

    @task(2)
    def get_single_detection(self):
        """Get a specific detection — fast Redis lookup."""
        detection_id = f"det-{random.randint(1, 100)}"
        with self.client.get(
            f"/api/v1/detections/{detection_id}",
            headers=self.headers,
            name="/api/v1/detections/{id} [GET]",
            catch_response=True
        ) as response:
            if response.status_code in (200, 404):
                response.success()   # 404 is expected for random IDs
            else:
                response.failure(f"Unexpected status: {response.status_code}")

    @task(1)
    def run_detection(self):
        """Inference task — most expensive, lowest weight."""
        image_bytes = make_test_image()
        self.client.post(
            "/api/v1/detections?confidence=0.5",
            files={"file": ("test.jpg", image_bytes, "image/jpeg")},
            headers=self.headers,
            name="/api/v1/detections [POST]"
        )

    @task(1)
    def health_check(self):
        """Health check — should always be fast."""
        self.client.get("/health", name="/health")


class AdminUser(HttpUser):
    """
    Simulates an admin user doing management tasks.
    Much lower weight — fewer admins than regular users.
    """
    wait_time = between(5, 15)
    weight = 1    # 1 admin per 10 regular users (SecuritySystemUser weight=10)

    def on_start(self):
        response = self.client.post("/auth/login", data={
            "username": "admin",
            "password": "adminpassword"
        })
        if response.status_code == 200:
            token = response.json()["access_token"]
            self.headers = {"Authorization": f"Bearer {token}"}
        else:
            self.headers = {}

    @task
    def list_cameras(self):
        self.client.get("/api/v1/cameras", headers=self.headers)

    @task
    def view_stats(self):
        self.client.get("/api/v1/stats", headers=self.headers)
'''

print("📄 tests/performance/locustfile.py:")
print(locustfile_py)

print("""
📋 RUNNING LOCUST:

  # Web UI mode (open browser at http://localhost:8089)
  locust -f tests/performance/locustfile.py --host=http://localhost:8000

  # Headless mode (for CI)
  locust -f tests/performance/locustfile.py \\
    --host=http://localhost:8000 \\
    --headless \\
    --users 50 \\           # 50 concurrent users
    --spawn-rate 5 \\       # add 5 users per second
    --run-time 60s \\       # run for 60 seconds
    --html results/load_test_report.html

  # Key metrics to watch:
  # RPS         → Requests per second (throughput)
  # p50         → 50th percentile response time (median)
  # p95         → 95th percentile (most users experience this)
  # p99         → 99th percentile (worst-case for most users)
  # Failures    → % of requests that failed
""")

print("\n✅ Exercise 4.1 Complete!")
print("=" * 80)


EXERCISE 4.1: Load Testing with Locust
📄 tests/performance/locustfile.py:

# ============================================================
# tests/performance/locustfile.py — Load Test Scenarios
# ============================================================

from locust import HttpUser, task, between, events
import io
import json
import random
from PIL import Image


def make_test_image(width=640, height=480) -> bytes:
    """Generate a test image for upload tasks."""
    img = Image.new("RGB", (width, height), color=(
        random.randint(0, 255),
        random.randint(0, 255),
        random.randint(0, 255)
    ))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


class SecuritySystemUser(HttpUser):
    """
    Simulates a typical Security System API user.
    wait_time: each user waits 1-3 seconds between requests
    (simulates realistic user behavior, not a DoS attack)
    """
    wait_time = between(1, 3)

    def on_start(self):
  

In [12]:
# ==================================================
# EXERCISE 4.2: INTERPRETING LOAD TEST RESULTS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.2: Interpreting Load Test Results")
print("=" * 80)

print("""
📊 EXAMPLE LOAD TEST RESULTS (50 users, 60 seconds):

  Endpoint                        RPS    p50    p95    p99    Fail%
  ───────────────────────────────────────────────────────────────────
  GET  /health                    45     8ms    15ms   22ms   0%
  GET  /api/v1/detections         38     35ms   80ms   120ms  0%
  GET  /api/v1/detections/{id}    22     12ms   45ms   90ms   2%
  POST /api/v1/detections         8      850ms  2.1s   4.2s   5%

📋 HOW TO READ THIS:

  GET /health:
  ✅ 45 RPS, 8ms median — excellent, as expected for a simple check

  GET /api/v1/detections:
  ✅ 38 RPS, 35ms median — Redis lookup is fast

  GET /api/v1/detections/{id}:
  ⚠️  2% failures — likely hitting nonexistent IDs (expected in load test)
      If real IDs: Redis connection pool may need tuning

  POST /api/v1/detections (ML inference):
  ⚠️  8 RPS, 850ms median, 4.2s p99, 5% failures
      → Model inference is the bottleneck (expected on CPU)
      → On GPU: p50 would drop to ~80ms
      → Failures likely rate limiting (10/min per user) kicking in

📋 PERFORMANCE TARGETS FOR ML APIs:

  Endpoint Type        p50 Target   p95 Target   Max Fail%
  ──────────────────────────────────────────────────────────
  Health check         < 20ms       < 50ms        0%
  Read (cached)        < 50ms       < 150ms       0%
  ML inference (CPU)   < 1s         < 3s          < 1%
  ML inference (GPU)   < 100ms      < 300ms       < 0.1%

📋 COMMON PERFORMANCE FIXES:

  Slow inference:
  → Cache results (identical inputs → Redis hit → skip inference)
  → Use smaller model (YOLOv8n vs YOLOv8x: 4x faster, ~5% less accurate)
  → Batch inference (process multiple frames together)
  → GPU deployment

  High p99 (tail latency):
  → Increase Uvicorn workers (but watch memory with ML models)
  → Move inference to async background job + polling
  → Implement request queue with max_concurrent limit

  Failures under load:
  → Check rate limiting config (may be too aggressive)
  → Check Redis connection pool size
  → Check Docker memory limits (OOMKilled under load)
""")

print("\n✅ Exercise 4.2 Complete!")
print("=" * 80)


EXERCISE 4.2: Interpreting Load Test Results

📊 EXAMPLE LOAD TEST RESULTS (50 users, 60 seconds):

  Endpoint                        RPS    p50    p95    p99    Fail%
  ───────────────────────────────────────────────────────────────────
  GET  /health                    45     8ms    15ms   22ms   0%
  GET  /api/v1/detections         38     35ms   80ms   120ms  0%
  GET  /api/v1/detections/{id}    22     12ms   45ms   90ms   2%
  POST /api/v1/detections         8      850ms  2.1s   4.2s   5%

📋 HOW TO READ THIS:

  GET /health:
  ✅ 45 RPS, 8ms median — excellent, as expected for a simple check

  GET /api/v1/detections:
  ✅ 38 RPS, 35ms median — Redis lookup is fast

  GET /api/v1/detections/{id}:
  ⚠️  2% failures — likely hitting nonexistent IDs (expected in load test)
      If real IDs: Redis connection pool may need tuning

  POST /api/v1/detections (ML inference):
  ⚠️  8 RPS, 850ms median, 4.2s p99, 5% failures
      → Model inference is the bottleneck (expected on CPU)
      → 

In [13]:
# ==================================================
# EXERCISE 4.3: COMPLETE TEST SUITE SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.3: Complete Test Suite Structure")
print("=" * 80)

print("""
📂 Final test suite structure for the Security System:

  week4_detection_tracking/
  └── tests/
      ├── __init__.py
      ├── conftest.py                        ← Shared fixtures (session scope)
      ├── fixtures/
      │   └── test_image.jpg                 ← Small test image
      │
      ├── unit/                              ← pytest -m unit (fast, no deps)
      │   ├── test_schemas.py                ← Pydantic model validation
      │   ├── test_jwt.py                    ← Token creation/verification
      │   └── test_api_key.py                ← Key hashing, compare_digest
      │
      ├── integration/                       ← pytest -m integration
      │   ├── test_integration_redis.py      ← Service + fakeredis
      │   ├── test_detections.py             ← Full endpoint tests
      │   ├── test_auth.py                   ← Auth flow tests
      │   └── test_cameras.py                ← Camera endpoint tests
      │
      └── performance/                       ← Run separately (not in CI)
          ├── locustfile.py                  ← Load test scenarios
          └── results/                       ← HTML reports (git-ignored)

📋 RUNNING THE TEST SUITE:

  # All tests
  pytest tests/

  # Only unit tests (fast — no external deps)
  pytest tests/ -m unit

  # Only integration tests
  pytest tests/ -m integration

  # Skip slow/performance tests
  pytest tests/ -m "not slow and not performance"

  # With coverage
  pytest tests/ -m "unit or integration" \\
    --cov=api --cov-report=term-missing --cov-fail-under=80

  # Performance tests (run separately, needs running server)
  locust -f tests/performance/locustfile.py \\
    --host=http://localhost:8000 --headless \\
    --users 50 --spawn-rate 5 --run-time 60s
""")

print("\n✅ Exercise 4.3 Complete!")
print("=" * 80)


EXERCISE 4.3: Complete Test Suite Structure

📂 Final test suite structure for the Security System:

  week4_detection_tracking/
  └── tests/
      ├── __init__.py
      ├── conftest.py                        ← Shared fixtures (session scope)
      ├── fixtures/
      │   └── test_image.jpg                 ← Small test image
      │
      ├── unit/                              ← pytest -m unit (fast, no deps)
      │   ├── test_schemas.py                ← Pydantic model validation
      │   ├── test_jwt.py                    ← Token creation/verification
      │   └── test_api_key.py                ← Key hashing, compare_digest
      │
      ├── integration/                       ← pytest -m integration
      │   ├── test_integration_redis.py      ← Service + fakeredis
      │   ├── test_detections.py             ← Full endpoint tests
      │   ├── test_auth.py                   ← Auth flow tests
      │   └── test_cameras.py                ← Camera endpoint tests
      │
      └── per

In [14]:
print("\n" + "=" * 80)
print("🎯 DAY 97 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Advanced pytest patterns: fixture scopes, teardown, session vs function
   ✅ Parametrized tests for confidence validation, file types, and RBAC
   ✅ Custom markers (unit, integration, auth, slow, performance)
   ✅ Test coverage with pytest-cov: .coveragerc, 80% target, HTML report
   ✅ Redis integration tests with fakeredis (cache, TTL, list, delete)
   ✅ Locust load testing: locustfile.py with weighted tasks and user roles
   ✅ Load test result interpretation and performance targets for ML APIs
   ✅ Complete test suite structure finalized

💡 KEY FILES CREATED TODAY:
   tests/
   ├── conftest.py                   ← Advanced fixtures (fakeredis, role tokens)
   ├── unit/
   │   ├── test_schemas.py
   │   ├── test_jwt.py
   │   └── test_api_key.py
   ├── integration/
   │   ├── test_integration_redis.py
   │   ├── test_detections.py (parametrized)
   │   └── test_auth.py (RBAC matrix)
   └── performance/
       └── locustfile.py
   .coveragerc                       ← Coverage configuration
   pytest.ini                        ← Pytest configuration + markers

🎯 TOMORROW (DAY 98) — FINAL DAY OF WEEK 14:
   - Model monitoring fundamentals
   - Structured logging with Loguru
   - Prometheus metrics for the FastAPI API
   - Grafana dashboard setup
   - Alerting on model performance degradation

Week 14 Progress: 85.7% (6/7 days complete)
Overall Progress: Day 97/168 (57.7% complete)
""")

print("=" * 80)


🎯 DAY 97 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Advanced pytest patterns: fixture scopes, teardown, session vs function
   ✅ Parametrized tests for confidence validation, file types, and RBAC
   ✅ Custom markers (unit, integration, auth, slow, performance)
   ✅ Test coverage with pytest-cov: .coveragerc, 80% target, HTML report
   ✅ Redis integration tests with fakeredis (cache, TTL, list, delete)
   ✅ Locust load testing: locustfile.py with weighted tasks and user roles
   ✅ Load test result interpretation and performance targets for ML APIs
   ✅ Complete test suite structure finalized

💡 KEY FILES CREATED TODAY:
   tests/
   ├── conftest.py                   ← Advanced fixtures (fakeredis, role tokens)
   ├── unit/
   │   ├── test_schemas.py
   │   ├── test_jwt.py
   │   └── test_api_key.py
   ├── integration/
   │   ├── test_integration_redis.py
   │   ├── test_detections.py (parametrized)
   │   └── test_auth.py (RBAC matrix)
   └── performance/
       └── locustfile.py
   .covera